# Distance Metrics Analysis (ETL Pipeline)

Extracts, transforms, and analyzes distance metric tables (`*distance_metrics_*.csv`) produced by `notebooks/single_region_full_validation.ipynb` across multiple single-region validation runs.

### Pipeline Highlights:
- Configurable input paths to target validation execution folders.
- Dynamic extraction of kinematic region `(ET, ETA)` from filenames.
- Lineage tracking of origin `ExecutionFolder`.
- Consolidated multi-region analysis and export.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mplhep as mh

mh.style.use("ATLAS")

from src.core.Validation.DistanceMetricsETL import DistanceMetricsETL

print(f"Project root: {PROJECT_ROOT}")

## 1. ETL Configuration

Specify the validation folders containing the `Tables/` directory, or provide folder paths directly.

In [ ]:
# Configure folders to scan. Example:
# VALIDATION_FOLDERS = [
#     PROJECT_ROOT / "notebooks/SingleRegionValidation_config_nameModelV9_HighBatch_100Rings_id20261029001623",
# ]
VALIDATION_FOLDERS: list[Path] = []

# Auto-discover all SingleRegionValidation folders if list is empty
if not VALIDATION_FOLDERS:
    discovered = sorted(list((PROJECT_ROOT / "notebooks").glob("SingleRegionValidation_*")))
    VALIDATION_FOLDERS = [d for d in discovered if d.is_dir()]

FILE_PATTERN: str = "*distance_metrics_*.csv"

print(f"Configured {len(VALIDATION_FOLDERS)} folders to scan:")
for folder in VALIDATION_FOLDERS:
    print(f" - {folder}")

## 2. Execute Extraction & Consolidation

In [ ]:
etl = DistanceMetricsETL(folder_paths=VALIDATION_FOLDERS, file_pattern=FILE_PATTERN)
print(f"Discovered {len(etl.discovered_files)} distance metric CSV files.")

df = etl.extract_and_consolidate()
print(f"Consolidated DataFrame shape: {df.shape}")
display(df.head())

## 3. Executive Summary by Region & Run

In [ ]:
summary_df = etl.get_summary(df)
if not summary_df.empty:
    display(summary_df)
else:
    print("No records found to summarize.")

## 4. Query & Inspect Specific Metrics & Features

In [ ]:
if not df.empty:
    print("Available Metric Types:", df["MetricType"].unique())
    print("Available Features:", df["Feature"].unique() if "Feature" in df.columns else "N/A")

    # Example pivot / inspection
    metric_type_target = df["MetricType"].unique()[0]
    subset = df[df["MetricType"] == metric_type_target]
    print(f"\nSample data for {metric_type_target}:")
    display(subset.head())
else:
    print("Consolidated DataFrame is empty.")